In [ ]:
from src.data_loader import load_split_from_local_files

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
import pandas as pd
from tqdm import tqdm

import evaluate
from sklearn.metrics import classification_report, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
model_path = "./model"  # change if needed

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Model loaded")

In [ ]:
path_to_test = "DialoGPT/sample_data/Test"

test_data = load_split_from_local_files(path_to_test, "test")

print("Test size:", len(test_data))

In [ ]:
emotion_map = {
    0: 'no emotion',
    1: 'anger',
    2: 'disgust',
    3: 'fear',
    4: 'happiness',
    5: 'sadness',
    6: 'surprise'
}

In [ ]:
predictions = []
references = []

for dialogue in tqdm(test_data):

    if len(dialogue["dialog"]) < 2:
        continue

    prompt = dialogue["dialog"][0]
    true_response = dialogue["dialog"][1]

    target_emotion = dialogue["emotion"][1]
    emotion_token = f"<|{emotion_map[target_emotion]}|>"

    input_str = f"{emotion_token} {prompt} {tokenizer.eos_token}"
    inputs = tokenizer(input_str, return_tensors="pt")

    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

    output = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_length=100,
        pad_token_id=tokenizer.eos_token_id
    )

    decoded = tokenizer.decode(output[0], skip_special_tokens=True)
    response = decoded.replace(prompt, "").strip()

    predictions.append(response)
    references.append(true_response)

print("Generated:", len(predictions))
print("References:", len(references))

In [ ]:
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

In [ ]:
rouge_results = rouge.compute(predictions=predictions, references=references)

print("ROUGE Results:")
print(rouge_results)

In [ ]:
bert_results = bertscore.compute(predictions=predictions, references=references, lang="en")

avg_precision = sum(bert_results["precision"]) / len(bert_results["precision"])
avg_recall = sum(bert_results["recall"]) / len(bert_results["recall"])
avg_f1 = sum(bert_results["f1"]) / len(bert_results["f1"])

print("\nBERTScore:")
print(f"Precision: {avg_precision:.4f}")
print(f"Recall: {avg_recall:.4f}")
print(f"F1: {avg_f1:.4f}")

In [ ]:
emotion_classifier = pipeline(
    "text-classification",
    model="SamLowe/roberta-base-go_emotions",
    top_k=1
)

print("Emotion classifier ready")

In [ ]:
classifier_to_our_map = {
    'sadness': 5,
    'anger': 1,
    'surprise': 6,
    'fear': 3,
    'joy': 4,
    'neutral': 0
}

In [ ]:
generated_emotions = []
actual_emotions = []

for dialogue in tqdm(test_data):

    if len(dialogue["dialog"]) < 2:
        continue

    prompt = dialogue["dialog"][0]

    input_str = f"<|no emotion|> {prompt} {tokenizer.eos_token}"
    inputs = tokenizer(input_str, return_tensors="pt")

    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

    output = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_length=100,
        pad_token_id=tokenizer.eos_token_id
    )

    decoded = tokenizer.decode(output[0], skip_special_tokens=True)
    response = decoded.replace(prompt, "").strip()

    if response:
        pred_label = emotion_classifier(response)[0][0]["label"]
        pred_idx = classifier_to_our_map.get(pred_label, 0)

        generated_emotions.append(pred_idx)
        actual_emotions.append(dialogue["emotion"][1])

In [ ]:
report = classification_report(
    actual_emotions,
    generated_emotions,
    target_names=list(emotion_map.values())
)

print(report)

In [ ]:
report_dict = classification_report(
    actual_emotions,
    generated_emotions,
    target_names=list(emotion_map.values()),
    output_dict=True
)

df = pd.DataFrame(report_dict).T.iloc[:-3]

df[['precision','recall','f1-score']].plot(kind='bar', figsize=(10,6))

plt.title("Precision / Recall / F1 per Emotion")
plt.ylim(0,1)
plt.tight_layout()
plt.show()

In [ ]:
summary = pd.Series({
    "Accuracy": report_dict["accuracy"],
    "Macro F1": report_dict["macro avg"]["f1-score"],
    "Weighted F1": report_dict["weighted avg"]["f1-score"]
})

summary.plot(kind="bar", figsize=(6,4))
plt.ylim(0,1)
plt.title("Overall Model Performance")
plt.show()

In [ ]:
labels_idx = list(sorted(emotion_map.keys()))
labels_txt = [emotion_map[i] for i in labels_idx]

cm = confusion_matrix(actual_emotions, generated_emotions, labels=labels_idx)

plt.figure(figsize=(7,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels_txt, yticklabels=labels_txt)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()